# ARENA3 정책 반복 1라운드 — 자료 모으기, 조각 2 / 3

**이 파일 하나 = 런타임 하나입니다.** Colab은 같은 노트북 파일을 탭 여러 개로 열어도 런타임 하나에 붙어서, 조각마다 파일을 따로 둡니다.
- 조각 0 = `ARENA3_PI_R1_data.ipynb`, 조각 1 = `ARENA3_PI_R1_data_shard1.ipynb`, 조각 2 = `ARENA3_PI_R1_data_shard2.ipynb`
- 세 파일을 하나씩 열어(런타임 유형 **CPU + 높은 RAM**) 각각 **런타임 → 모두 실행**을 누르세요. 첫 칸의 `SHARD`는 이미 2로 정해져 있으니 바꾸지 마세요.
- 같은 조각을 두 런타임에서 동시에 돌리지 마세요(같은 결과 파일을 서로 덮어씁니다).

하는 일과 순서는 조각 0 파일과 같습니다: 실행 전 점검(약 15분) → 자료 모으기(약 15~17시간) → 진행 확인.
끊기면 다시 **모두 실행**하면 끝난 판은 건너뜁니다. 마지막 칸에 **64/64**가 나오면(세 조각이 다 끝난 것) `ARENA3_PI_R1_learn`을 실행하세요.

In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, json

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/FutureTraffic")
ARCHIVE = DRIVE / "FutureTraffic_ARENA3_PI.zip"
A3 = DRIVE / "arena3"
OUT = A3 / "pi" / "R1"
WORKSPACE = Path("/content/FutureTraffic_ARENA3_PI")
MIN_CPUS = 8
assert ARCHIVE.is_file(), f"묶음이 없습니다: {ARCHIVE}"
CPUS = os.cpu_count() or 1

SHARD = 2          # 이 파일 = 조각 2(바꾸지 마세요)
SHARDS = 3         # 모든 런타임에서 같은 값
BASE_ROOT = A3 / "stage2" / "S2A_train"      # 지금 정책 S2A_pre(난수 42)가 있는 곳
assert 0 <= SHARD < SHARDS, "SHARD는 0 이상 SHARDS 미만이어야 합니다."
print("CPU 개수:", CPUS, "| 조각:", SHARD, "/", SHARDS, "| 결과 폴더:", OUT / "data")
assert CPUS >= MIN_CPUS, (f"CPU가 {CPUS}개입니다. 런타임 → 런타임 유형 변경에서 CPU와 '높은 RAM'을 고른 뒤 다시 실행하세요. "
                          "(롤아웃을 CPU 수만큼 동시에 돌립니다. CPU 2개면 4배 오래 걸려 24시간 한도에 걸립니다.)")
assert (BASE_ROOT / "seed_42" / "pretrained" / "model.pt").exists(), f"지금 정책 모델이 없습니다: {BASE_ROOT}"

In [ ]:
# ZIP 전체의 파일 목록·SHA-256을 확인한 뒤 검증된 전용 추출기를 사용합니다.
import hashlib, stat, types, zipfile
from pathlib import PurePosixPath

with zipfile.ZipFile(ARCHIVE) as archive:
    names = archive.namelist()
    assert len(names) == len(set(names)) and names.count("manifest.json") == 1, "ZIP 파일 목록 오류"
    package_manifest = json.loads(archive.read("manifest.json"))
    assert package_manifest.get("kind") == "WEIGHT1_PACKAGE_V1", "검증할 수 없는 ZIP 형식입니다"
    assert package_manifest.get("experiment") == "ARENA3_PI_R1", "ARENA3 정책 반복 1라운드 전용 ZIP이 아닙니다"
    assert package_manifest.get("source_models_included") is False, "모델·결과가 든 묶음은 쓰지 않습니다"
    expected = package_manifest["files"]
    assert expected and "manifest.json" not in expected and set(names) == set(expected) | {"manifest.json"}, "ZIP과 manifest 목록 불일치"
    for info in archive.infolist():
        name, path = info.filename, PurePosixPath(info.filename)
        assert path.parts and not path.is_absolute() and ".." not in path.parts and path.as_posix() == name, "잘못된 ZIP 경로"
        assert "\\" not in name and not info.is_dir() and not stat.S_ISLNK(info.external_attr >> 16), "잘못된 ZIP 파일 형식"
        if name != "manifest.json":
            assert hashlib.sha256(archive.read(name)).hexdigest() == expected[name], f"SHA-256 불일치: {name}"
    source_code = archive.read("src/explore_weight_package.py").decode("utf-8")

bootstrap = types.ModuleType("arena3_verified_package")
bootstrap.__file__ = str(WORKSPACE / "src/explore_weight_package.py")
exec(compile(source_code, bootstrap.__file__, "exec"), bootstrap.__dict__)
package_manifest = bootstrap.extract_verified(ARCHIVE, WORKSPACE)
print(f"검증·추출 완료: {len(package_manifest['files'])}개 파일 → {WORKSPACE} | 잠금 {package_manifest['lock_sha256'][:12]} "
      f"| 정책 반복 설정 {package_manifest['pi_config_sha256'][:12]}")

In [ ]:
import subprocess, importlib.metadata

os.environ["FOCUSED_RULESET"] = "pet1"
os.environ["FOCUSED_NORTH_EXT"] = "240"
EXPECTED = {"numpy": "2.1.3", "torch": "2.11.0+cpu", "stable-baselines3": "2.9.0",
            "sumo-rl": "1.4.5", "eclipse-sumo": "1.27.1"}

def installed_versions():
    versions = {}
    for name in EXPECTED:
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = None
    return versions

before = installed_versions()
if before == EXPECTED:
    print("이전 실험과 패키지 버전이 같아 재설치를 건너뜁니다.")
else:
    print("이전 실험과 같은 패키지 버전을 준비합니다:", before)
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], cwd=WORKSPACE, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "numpy==2.1.3", "stable-baselines3==2.9.0", "eclipse-sumo==1.27.1"],
                   cwd=WORKSPACE, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "torch==2.11.0+cpu", "--index-url", "https://download.pytorch.org/whl/cpu"],
                   cwd=WORKSPACE, check=True)
after = installed_versions()
assert after == EXPECTED, f"패키지 버전이 이전 실험과 다릅니다(평가 지문이 달라져 기준선과 비교할 수 없어 멈춥니다): {after}"
print("실행 패키지 검증 통과:", after)

In [ ]:
sys.path.insert(0, str(WORKSPACE / "src"))
import arena3_colab as colab
from IPython.display import display, Markdown

colab.run_live(["-m", "unittest", "discover", "-s", "tests", "-p", "test_arena3*.py"], WORKSPACE)
print("실행 전 검사(SUMO 없이) 통과")

In [ ]:
# 1) 실행 전 점검(약 15분): S2A_pre로 학습 상황 하나를 그냥 돌린 것과, 상태를 저장·롤아웃하며 돌린 것이 소수점까지 같은지,
#    롤아웃 복원이 맞는지, 판당 시간이 얼마인지 봅니다. 결과는 이 런타임 전용 점검 폴더에만 씁니다.
SMOKE = OUT / "smoke" / f"shard{SHARD}"
previous = colab.read_json(SMOKE / "smoke.json")
if previous and previous.get("passed"):
    print("이전 점검이 통과해 건너뜁니다. 판당 약", round(previous["estimate_cpu_hours_per_episode"], 2), "CPU-시간")
else:
    colab.run_live(["src/arena3_pi.py", "smoke", "--out", str(SMOKE), "--work", "/content/pi_smoke",
                    "--base-root", str(BASE_ROOT), "--workers", str(CPUS)], WORKSPACE)
    s = colab.read_json(SMOKE / "smoke.json")
    print("점검 통과. 판당 약", round(s["estimate_cpu_hours_per_episode"], 2), "CPU-시간, 64판 합 약",
          round(s["estimate_cpu_hours_all"]), "CPU-시간")

In [ ]:
# 2) 자료 모으기: 이 런타임 몫(64판 중 SHARD번째부터 SHARDS칸씩). 셀을 멈춰도 계속 돕니다.
#    런타임이 끊기면 처음 네 칸 → 이 칸을 다시 실행하세요. 끝난 판은 건너뜁니다.
assert "colab" in globals(), "처음 네 칸(1~4)을 먼저 실행하세요. 런타임이 끊기면 변수와 작업 폴더가 사라집니다."
DATA = OUT / "data"
LOGS = OUT / "logs"
NAME = f"collect_shard{SHARD}"


def describe(name):
    s = colab.read_json(DATA / "status" / f"{name}.json")
    if not s:
        return None
    return (f"끝난 판 {s['done']}/{s['total']}, 롤아웃 중 {len(s['inflight'])}판, 실패 {len(s['failed'])}, "
            f"복원 불일치 {s['restore_mismatched']}/{s['restore_checked']}")


pids = colab.launch({NAME: ["src/arena3_pi.py", "collect", "--out", str(DATA), "--work", "/content/pi",
                            "--shard", str(SHARD), "--shards", str(SHARDS), "--base-root", str(BASE_ROOT),
                            "--workers", str(CPUS)]}, WORKSPACE, LOGS)
code = colab.watch(pids, LOGS, 600, describe)
assert code == 0, "자료 모으기에서 오류가 났습니다. 위 로그를 저에게 보내 주세요(다시 실행하면 끝난 판은 건너뜁니다)."

In [ ]:
# 3) 모든 런타임의 진행(언제든 다시 실행해 볼 수 있습니다).
for f in sorted((OUT / "data" / "status").glob("collect_shard*.json")):
    s = colab.read_json(f) or {}
    print(f.stem, f"{s.get('done')}/{s.get('total')}", "| 실패", len(s.get("failed", [])),
          "| 복원 불일치", f"{s.get('restore_mismatched')}/{s.get('restore_checked')}")
done = len(list((OUT / "data" / "runs" / "PI_DATA").glob("*/cache.json")))
print(f"끝난 판 {done}/64")
if done == 64:
    print("64판이 모두 끝났습니다. ARENA3_PI_R1_learn 노트북을 실행하세요.")